In [1]:
# -*- coding: utf-8 -*-
"""GAN-GRU 因子在沪深300上的指数增强回测测试。

组合构建使用“自定义指数倾斜函数”：先对 GAN-GRU 分数做截面 Z-score，
再使用受上下限约束的指数函数生成非负倾斜乘数，并乘以沪深300历史权重。

时序口径：信号日收盘后计算因子，下一交易日开盘执行；每20个交易日调仓。
"""

import copy
import sys
from importlib import import_module

import numpy as np
import pandas as pd

from factor_lib.function.bigquant_function.strategies.index_enhancement_backtest import (
    run_index_enhancement_backtest,
)


sys.dont_write_bytecode = True


# =============================================================================
# 1. 回测参数
# =============================================================================

START_DATE = "2023-04-28"
END_DATE = "2024-08-29"
BENCHMARK = "000300.SH"
REBALANCE_INTERVAL = 20
INITIAL_CASH = 10_000_000


# =============================================================================
# 2. GAN-GRU 滚动训练参数
# =============================================================================

# feature_spec=None 表示使用 gan_gru_score.py 中登记的20项默认特征。
# 标签期限和训练/验证集 purge 间隔均设为20个交易日，与本策略持有期一致。
TRAINING_CONFIG = {
    "sequence_length": 40,
    "training_window_days": 200,
    "retrain_interval_days": 100,
    "label_horizon_days": REBALANCE_INTERVAL,
    "validation_ratio": 0.20,
    "purge_trading_days": REBALANCE_INTERVAL,
    "minimum_validation_dates": 20,
    "minimum_rankic_stocks": 30,
    "minimum_training_samples": 1000,
    "latent_dim": 10,
    "discriminator_hidden_size": 32,
    "lambda_reconstruction": 0.5,
    "gan_epochs": 4,
    "gru_max_epochs": 20,
    "early_stop_patience": 4,
    "rankic_min_delta": 0.0001,
    "batch_size": 1024,
    # 本次测试显式使用4个CPU线程。
    "cpu_threads": 4,
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.20,
    "gan_learning_rate": 0.0002,
    "gru_learning_rate": 0.001,
    "random_seed": 42,
}


# =============================================================================
# 3. 自定义指数倾斜函数
# =============================================================================

def bounded_exponential_tilt(
    transformed_scores,
    strength=0.35,
    minimum_multiplier=0.70,
    maximum_multiplier=1.50,
):
    """将标准化因子分数转换为有限、非负的指数倾斜乘数。

    该函数只负责产生相对基准的乘数：

        multiplier = exp(strength * transformed_score)

    随后将乘数限制在 [minimum_multiplier, maximum_multiplier]，避免单次
    模型分数异常造成过度偏离。最终权重归一化由策略函数统一完成。
    """
    if not isinstance(transformed_scores, pd.Series):
        raise TypeError("transformed_scores 必须是 pandas.Series。")

    strength = float(strength)
    minimum_multiplier = float(minimum_multiplier)
    maximum_multiplier = float(maximum_multiplier)

    if not np.isfinite(strength) or strength < 0:
        raise ValueError("strength 必须是有限非负数。")
    if (
        not np.isfinite(minimum_multiplier)
        or not np.isfinite(maximum_multiplier)
        or minimum_multiplier < 0
        or minimum_multiplier > maximum_multiplier
        or maximum_multiplier <= 0
    ):
        raise ValueError("倾斜乘数上下限设置无效。")

    multipliers = np.exp(strength * transformed_scores.astype(float))
    multipliers = multipliers.clip(
        lower=minimum_multiplier,
        upper=maximum_multiplier,
    )

    if multipliers.isna().any() or not np.isfinite(multipliers).all():
        raise ValueError("自定义倾斜函数产生了非有限值。")
    return multipliers


# =============================================================================
# 4. 建立本次回测独立使用的滚动模型状态提供器
# =============================================================================

gan_gru_module = import_module(
    "factor_lib.Factor Repository.machine_learning_factors.gan_gru_score"
)

# 回测仅使用内存状态；每次重新运行本单元格都会建立一个新的模型时间线。
model_state_provider = gan_gru_module.build_model_state_provider(
    persistence_mode="memory",
)

FACTOR_PARAMS = {
    "feature_spec": None,
    "model_state_provider": model_state_provider,
    "training_config": copy.deepcopy(TRAINING_CONFIG),
}


# =============================================================================
# 5. 运行沪深300指数增强回测
# =============================================================================

print("[GAN-GRU指数增强测试] 基准指数：", BENCHMARK)
print("[GAN-GRU指数增强测试] 默认基础特征数：20")
print("[GAN-GRU指数增强测试] 调仓间隔：20个交易日")
print("[GAN-GRU指数增强测试] 倾斜函数：受限指数倾斜")
print("[GAN-GRU指数增强测试] 滚动训练参数：", TRAINING_CONFIG)

backtest_result = run_index_enhancement_backtest(
    start_date=START_DATE,
    end_date=END_DATE,
    reference_portfolio={
        "type": "index",
        "index_code": BENCHMARK,
    },
    factor_name="gan_gru_score",
    factor_params=FACTOR_PARAMS,
    # GAN-GRU 分数越高，表示模型预测的未来收益越高。
    signal_direction=1,
    construction_method="benchmark_tilt",
    construction_params={
        # 策略会先将当前截面的模型分数标准化，再交给自定义函数。
        "score_transform": "zscore",
        "score_clip": 3.0,
        "tilt_function": "custom",
        "custom_tilt_function": bounded_exponential_tilt,
        "custom_tilt_params": {
            "strength": 0.35,
            "minimum_multiplier": 0.70,
            "maximum_multiplier": 1.50,
        },
    },
    rebalance_rule={
        "type": "fixed_interval",
        "interval_trading_days": REBALANCE_INTERVAL,
    },
    portfolio_constraints={
        "target_stock_exposure": 1.00,
        # 不允许策略层主动降低目标股票仓位；交易受阻仍可能形成实际现金。
        "min_stock_exposure": 1.00,
        # 以下可选组合约束全部关闭。
        "max_stock_weight": None,
        "max_active_weight": None,
        "max_turnover": None,
        "industry_active_weight_limit": None,
        "style_active_exposure_limit": None,
        "max_tracking_error": None,
    },
    risk_model={
        # 保留明确的行业分类口径；由于行业偏离约束已关闭，本次不会
        # 因该设置额外执行行业约束求解。
        "industry_scheme": "sw2021_l1",
        "style_fields": [],
    },
    feasibility_policy={
        # 没有可选组合约束需要放宽，因此使用严格模式。
        "mode": "strict",
        "stop_at_first_feasible": True,
        "final_action": "hold_previous",
    },
    execution_config={
        "order_price_field_buy": "open",
        "order_price_field_sell": "open",
        "volume_limit": 0.025,
        "slippage_value": 0.001,
        "weight_tolerance": 0.0005,
        # 本测试要求严格保持20日频；成分变化只在下个固定调仓日纳入。
        "rebalance_on_index_reconstitution": False,
    },
    trading_costs={
        "buy_cost": 0.0003,
        "sell_cost": 0.0003,
        "min_cost": 5.0,
        "tax_ratio": 0.0005,
    },
    initial_cash=INITIAL_CASH,
    performance_benchmark=BENCHMARK,
    show_progress=True,
    progress_every=1,
)


# BigTrader 回测图表会由策略函数自动显示。
# 如果需要检查模型信号、约束放宽和实际成交，可按需取消以下注释：
# from IPython.display import display
# display(backtest_result["data_diagnostics"])
# display(backtest_result["rebalance_audit"])
# display(backtest_result["feasibility_audit"])
# display(backtest_result["risk_audit"])
# display(backtest_result["execution_audit"].head())
# display(backtest_result["trade_audit"].head())


[GAN-GRU指数增强测试] 基准指数： 000300.SH
[GAN-GRU指数增强测试] 默认基础特征数：20
[GAN-GRU指数增强测试] 调仓间隔：20个交易日
[GAN-GRU指数增强测试] 倾斜函数：受限指数倾斜
[GAN-GRU指数增强测试] 滚动训练参数： {'sequence_length': 40, 'training_window_days': 200, 'retrain_interval_days': 100, 'label_horizon_days': 20, 'validation_ratio': 0.2, 'purge_trading_days': 20, 'minimum_validation_dates': 20, 'minimum_rankic_stocks': 30, 'minimum_training_samples': 1000, 'latent_dim': 10, 'discriminator_hidden_size': 32, 'lambda_reconstruction': 0.5, 'gan_epochs': 4, 'gru_max_epochs': 20, 'early_stop_patience': 4, 'rankic_min_delta': 0.0001, 'batch_size': 1024, 'cpu_threads': 4, 'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2, 'gan_learning_rate': 0.0002, 'gru_learning_rate': 0.001, 'random_seed': 42}
[BigQuant loader] 20/20（100.00%），依赖数据加载完成，当前 volume_ratio_5d，121,829 行，耗时 3.8s                                                                                                                                                                  
[BigQuant 日频适配器] [3/3] 查询

[2026-08-14 19:17:24] [info     ] bigtrader.v35 运行完成 [575.261s].
[指数增强回测] [9/9] 回测与审计结果整理完成 | 1/1 (100.0%) | 订单3,890条，成交1,945条 | 已耗时 584.7s                                                                                                                                                                      
